# Vial Reaction-Diffusion Interpolation

Uses Gray-Scott reaction-diffusion to organically morph between captured vial states.
Instead of linear pixel lerp, the transition is a living chemical reaction.

**Input:** 49 vial captures (8 frames each = 392 frames)
**Output:** Reaction-diffusion interpolated transition frames + WebGL shader

Guinea Pig Trench LLC

In [ ]:
#@title 1. Setup + Upload vial sprite sheet
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import os, json, time

# Upload sprite sheet or use Kaggle dataset
SHEET_PATH = None

# Try Kaggle dataset first
kaggle_path = '/kaggle/input/guinea-pig-trench-vials/vial_spritesheet.png'
drive_path = '/content/drive/MyDrive/Guinea Pig Trench/sprites/vials/vial_spritesheet.png'

if os.path.exists(kaggle_path):
    SHEET_PATH = kaggle_path
    print(f'Found sprite sheet in Kaggle dataset')
elif os.path.exists(drive_path):
    SHEET_PATH = drive_path
    print(f'Found sprite sheet on Drive')
else:
    # Upload manually
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        SHEET_PATH = name
        break

# Load and parse sprite sheet
sheet = np.array(Image.open(SHEET_PATH))
FRAME_W, FRAME_H = 64, 100
COLS = 32
FRAMES_PER_CAPTURE = 8

total_frames = (sheet.shape[0] // FRAME_H) * COLS
# Trim to actual count
n_captures = total_frames // FRAMES_PER_CAPTURE

def get_frame(idx):
    col = idx % COLS
    row = idx // COLS
    y = row * FRAME_H
    x = col * FRAME_W
    return sheet[y:y+FRAME_H, x:x+FRAME_W].astype(np.float32) / 255.0

# Extract all key frames (frame 0 of each capture)
key_frames = [get_frame(i * FRAMES_PER_CAPTURE) for i in range(n_captures)]
print(f'Loaded {n_captures} captures, {len(key_frames)} key frames')
print(f'Frame size: {FRAME_W}x{FRAME_H}')

# Show first 8 key frames
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i, ax in enumerate(axes):
    if i < len(key_frames):
        ax.imshow(key_frames[i])
    ax.axis('off')
    ax.set_title(f'Key {i}')
plt.suptitle('First 8 Key Frames (F0 of each capture)')
plt.tight_layout()
plt.show()

In [ ]:
#@title 2. Gray-Scott Reaction-Diffusion Engine (GPU via CuPy)
try:
    import cupy as cp
    GPU = True
    print('CuPy available — running on GPU')
except ImportError:
    cp = np
    GPU = False
    print('CuPy not available — running on CPU (slower)')

class GrayScottRD:
    """
    Gray-Scott Reaction-Diffusion system.
    
    Two chemicals A and B:
      A + 2B -> 3B  (autocatalysis)
      dA/dt = dA * laplacian(A) - A*B*B + f*(1-A)
      dB/dt = dB * laplacian(B) + A*B*B - (f+k)*B
    
    f = feed rate (how fast A replenishes)
    k = kill rate (how fast B decays)
    
    Different (f,k) values produce different patterns:
      - Spots, stripes, coral, mitosis, worms
      - Image-driven: f,k vary per-pixel based on source image
    """
    
    def __init__(self, width, height, dA=1.0, dB=0.5, dt=1.0):
        self.w = width
        self.h = height
        self.dA = dA
        self.dB = dB
        self.dt = dt
        
        # Chemical concentrations
        self.A = cp.ones((height, width), dtype=cp.float32)
        self.B = cp.zeros((height, width), dtype=cp.float32)
        
        # Seed B in center
        cy, cx = height // 2, width // 2
        r = min(width, height) // 6
        yy, xx = cp.mgrid[:height, :width]
        mask = ((xx - cx)**2 + (yy - cy)**2) < r**2
        self.B[mask] = 1.0
        
        # Parameter maps (can vary per-pixel)
        self.f_map = cp.full((height, width), 0.055, dtype=cp.float32)
        self.k_map = cp.full((height, width), 0.062, dtype=cp.float32)
    
    def set_params_from_image(self, img, f_range=(0.01, 0.07), k_range=(0.045, 0.07)):
        """
        Set f,k parameters per-pixel based on image luminance.
        Dark pixels -> one pattern, bright pixels -> another.
        The reaction-diffusion conforms to the image organically.
        """
        if img.ndim == 3:
            # Convert to luminance
            lum = 0.299 * img[:,:,0] + 0.587 * img[:,:,1] + 0.114 * img[:,:,2]
        else:
            lum = img
        
        lum = cp.asarray(lum, dtype=cp.float32)
        
        # Map luminance to f,k
        self.f_map = cp.asarray(f_range[0] + lum * (f_range[1] - f_range[0]))
        self.k_map = cp.asarray(k_range[0] + lum * (k_range[1] - k_range[0]))
    
    def interpolate_params(self, img_a, img_b, t, f_range=(0.01, 0.07), k_range=(0.045, 0.07)):
        """
        Interpolate f,k parameter maps between two images.
        t=0 -> img_a parameters, t=1 -> img_b parameters.
        The RD system morphs organically between states.
        """
        if img_a.ndim == 3:
            lum_a = 0.299 * img_a[:,:,0] + 0.587 * img_a[:,:,1] + 0.114 * img_a[:,:,2]
        else:
            lum_a = img_a
        if img_b.ndim == 3:
            lum_b = 0.299 * img_b[:,:,0] + 0.587 * img_b[:,:,1] + 0.114 * img_b[:,:,2]
        else:
            lum_b = img_b
        
        # Blend luminance maps
        lum = lum_a * (1 - t) + lum_b * t
        lum = cp.asarray(lum, dtype=cp.float32)
        
        self.f_map = cp.asarray(f_range[0] + lum * (f_range[1] - f_range[0]))
        self.k_map = cp.asarray(k_range[0] + lum * (k_range[1] - k_range[0]))
    
    def laplacian(self, grid):
        """3x3 discrete Laplacian with wrapping."""
        return (
            0.05 * cp.roll(cp.roll(grid, 1, 0), 1, 1) +  # top-left
            0.20 * cp.roll(grid, 1, 0) +                   # top
            0.05 * cp.roll(cp.roll(grid, 1, 0), -1, 1) +  # top-right
            0.20 * cp.roll(grid, 1, 1) +                   # left
            -1.0 * grid +                                   # center
            0.20 * cp.roll(grid, -1, 1) +                  # right
            0.05 * cp.roll(cp.roll(grid, -1, 0), 1, 1) +  # bottom-left
            0.20 * cp.roll(grid, -1, 0) +                  # bottom
            0.05 * cp.roll(cp.roll(grid, -1, 0), -1, 1)   # bottom-right
        )
    
    def step(self, n=1):
        """Run n simulation steps."""
        for _ in range(n):
            lapA = self.laplacian(self.A)
            lapB = self.laplacian(self.B)
            
            ABB = self.A * self.B * self.B
            
            self.A += (self.dA * lapA - ABB + self.f_map * (1 - self.A)) * self.dt
            self.B += (self.dB * lapB + ABB - (self.f_map + self.k_map) * self.B) * self.dt
            
            # Clamp
            self.A = cp.clip(self.A, 0, 1)
            self.B = cp.clip(self.B, 0, 1)
    
    def get_image(self, colormap='vial'):
        """
        Convert RD state to colored image.
        'vial' colormap: teal for A-dominant, pink for B-dominant.
        """
        A_np = cp.asnumpy(self.A) if GPU else self.A
        B_np = cp.asnumpy(self.B) if GPU else self.B
        
        if colormap == 'vial':
            # Teal (0, 210, 255) for A, Pink (255, 96, 160) for B
            r = (1 - B_np) * 0.04 + B_np * 1.0
            g = (1 - B_np) * 0.82 + B_np * 0.38
            b = (1 - B_np) * 1.0 + B_np * 0.63
            
            # Darken based on A (background)
            brightness = 0.08 + A_np * 0.15 + B_np * 0.8
            r *= brightness
            g *= brightness
            b *= brightness
        else:
            r = 1 - B_np
            g = 1 - B_np * 0.5
            b = 1 - A_np * 0.3
        
        img = np.stack([r, g, b], axis=-1)
        return np.clip(img, 0, 1)

# Test
rd = GrayScottRD(FRAME_W * 2, FRAME_H * 2)  # 2x resolution
rd.set_params_from_image(key_frames[0][:,:,:3])

print('Running 500 steps...')
t0 = time.time()
rd.step(500)
print(f'Done in {time.time()-t0:.1f}s')

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
ax1.imshow(key_frames[0])
ax1.set_title('Source Key Frame')
ax1.axis('off')
ax2.imshow(rd.get_image(), interpolation='nearest')
ax2.set_title('RD State (500 steps)')
ax2.axis('off')
ax3.imshow(rd.get_image('grayscale'), interpolation='nearest')
ax3.set_title('RD Grayscale')
ax3.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#@title 3. Generate RD Interpolation Frames Between Key States
import imageio

# How many interpolation frames between each pair of key frames
INTERP_FRAMES = 16
# RD steps per interpolation frame
STEPS_PER_FRAME = 40
# How many capture pairs to process (start small)
N_PAIRS = min(8, n_captures - 1)

output_dir = '/kaggle/working/rd_frames' if os.path.exists('/kaggle') else 'rd_frames'
os.makedirs(output_dir, exist_ok=True)

all_rd_frames = []

for pair_idx in range(N_PAIRS):
    key_a = key_frames[pair_idx][:,:,:3]
    key_b = key_frames[pair_idx + 1][:,:,:3]
    
    print(f'\nPair {pair_idx}: Key {pair_idx} → Key {pair_idx+1}')
    
    # Fresh RD system for each pair
    rd = GrayScottRD(FRAME_W, FRAME_H)
    
    # Seed B from the source image — fluid regions get B=1
    lum_a = 0.299 * key_a[:,:,0] + 0.587 * key_a[:,:,1] + 0.114 * key_a[:,:,2]
    rd.B = cp.asarray(lum_a)
    
    # Warm up
    rd.set_params_from_image(key_a)
    rd.step(100)
    
    pair_frames = []
    
    for f in range(INTERP_FRAMES):
        t = f / (INTERP_FRAMES - 1)  # 0 to 1
        
        # Interpolate RD parameters between the two key frames
        rd.interpolate_params(key_a, key_b, t)
        
        # Run simulation
        rd.step(STEPS_PER_FRAME)
        
        # Capture frame
        rd_img = rd.get_image('vial')
        
        # Composite: blend RD pattern with linear interpolation of source images
        # RD provides the organic texture, source images provide the structure
        linear_blend = key_a * (1 - t) + key_b * t
        
        # 60% source structure + 40% RD organic texture
        composite = linear_blend * 0.6 + rd_img * 0.4
        composite = np.clip(composite, 0, 1)
        
        # Save frame
        frame_uint8 = (composite * 255).astype(np.uint8)
        frame_path = os.path.join(output_dir, f'rd_{pair_idx:02d}_{f:03d}.png')
        Image.fromarray(frame_uint8).save(frame_path)
        pair_frames.append(frame_uint8)
        
        if f % 4 == 0:
            print(f'  Frame {f}/{INTERP_FRAMES} (t={t:.2f})')
    
    all_rd_frames.extend(pair_frames)
    
    # Show this pair's transition
    fig, axes = plt.subplots(1, min(8, INTERP_FRAMES), figsize=(16, 3))
    step = max(1, INTERP_FRAMES // 8)
    for i, ax in enumerate(axes):
        idx = i * step
        if idx < len(pair_frames):
            ax.imshow(pair_frames[idx])
        ax.axis('off')
        ax.set_title(f't={idx/(INTERP_FRAMES-1):.1f}')
    plt.suptitle(f'RD Transition: Key {pair_idx} → Key {pair_idx+1}')
    plt.tight_layout()
    plt.show()

print(f'\nGenerated {len(all_rd_frames)} RD interpolation frames')

In [ ]:
#@title 4. Build RD Sprite Sheet + GIF Preview

# Build sprite sheet from RD frames
n_rd = len(all_rd_frames)
rd_cols = 32
rd_rows = (n_rd + rd_cols - 1) // rd_cols

rd_sheet = np.zeros((rd_rows * FRAME_H, rd_cols * FRAME_W, 3), dtype=np.uint8)
for i, frame in enumerate(all_rd_frames):
    col = i % rd_cols
    row = i // rd_cols
    # Handle RGBA vs RGB
    f = frame[:,:,:3] if frame.shape[2] == 4 else frame
    rd_sheet[row*FRAME_H:(row+1)*FRAME_H, col*FRAME_W:(col+1)*FRAME_W] = f

rd_sheet_path = os.path.join(output_dir, 'rd_spritesheet.png')
Image.fromarray(rd_sheet).save(rd_sheet_path)
print(f'RD sprite sheet: {rd_sheet.shape[1]}x{rd_sheet.shape[0]}, {os.path.getsize(rd_sheet_path)/1024:.0f} KB')

# GIF preview
gif_path = os.path.join(output_dir, 'rd_vial_preview.gif')
gif_frames = []

# Take every 2nd frame for smaller GIF
for i in range(0, len(all_rd_frames), 2):
    f = all_rd_frames[i]
    # Scale up for visibility
    pil_f = Image.fromarray(f[:,:,:3] if f.shape[2] == 4 else f)
    pil_f = pil_f.resize((FRAME_W * 3, FRAME_H * 3), Image.NEAREST)
    gif_frames.append(np.array(pil_f))

imageio.mimsave(gif_path, gif_frames, fps=12, loop=0)
print(f'GIF: {os.path.getsize(gif_path)/1024:.0f} KB, {len(gif_frames)} frames')

# Display GIF in notebook
from IPython.display import Image as IPImage
display(IPImage(filename=gif_path))

In [ ]:
#@title 5. Generate WebGL Shader for Portal

shader_code = '''
// Guinea Pig Trench — Reaction-Diffusion Vial Shader
// Gray-Scott RD with image-driven parameters
// Runs in the browser, transitions between vial states organically

precision highp float;

uniform sampler2D u_state;      // current RD state (A in R, B in G)
uniform sampler2D u_vialA;      // source vial image A
uniform sampler2D u_vialB;      // target vial image B  
uniform float u_t;              // transition progress 0..1
uniform vec2 u_resolution;
uniform float u_dA;             // diffusion rate A (default 1.0)
uniform float u_dB;             // diffusion rate B (default 0.5)

// Gray-Scott parameters driven by vial luminance
vec2 getFK(vec2 uv) {
    vec3 colA = texture2D(u_vialA, uv).rgb;
    vec3 colB = texture2D(u_vialB, uv).rgb;
    vec3 blended = mix(colA, colB, u_t);
    float lum = dot(blended, vec3(0.299, 0.587, 0.114));
    
    // Map luminance to f,k
    // Dark regions: mitosis pattern (f=0.028, k=0.062)
    // Bright regions: coral pattern (f=0.06, k=0.062)  
    float f = mix(0.028, 0.062, lum);
    float k = mix(0.058, 0.065, lum);
    return vec2(f, k);
}

void main() {
    vec2 uv = gl_FragCoord.xy / u_resolution;
    vec2 px = 1.0 / u_resolution;
    
    // Sample current state
    vec2 state = texture2D(u_state, uv).rg;
    float A = state.r;
    float B = state.g;
    
    // 3x3 Laplacian
    float lapA = 0.0, lapB = 0.0;
    for (int dy = -1; dy <= 1; dy++) {
        for (int dx = -1; dx <= 1; dx++) {
            vec2 neighbor = texture2D(u_state, uv + vec2(float(dx), float(dy)) * px).rg;
            float w = (dx == 0 && dy == 0) ? -1.0 : (dx == 0 || dy == 0) ? 0.2 : 0.05;
            lapA += neighbor.r * w;
            lapB += neighbor.g * w;
        }
    }
    
    // Gray-Scott equations
    vec2 fk = getFK(uv);
    float f = fk.x;
    float k = fk.y;
    
    float ABB = A * B * B;
    float newA = A + (u_dA * lapA - ABB + f * (1.0 - A));
    float newB = B + (u_dB * lapB + ABB - (f + k) * B);
    
    gl_FragColor = vec4(clamp(newA, 0.0, 1.0), clamp(newB, 0.0, 1.0), 0.0, 1.0);
}
'''

render_shader = '''
// Render RD state with vial color palette
precision highp float;

uniform sampler2D u_state;
uniform sampler2D u_vialA;
uniform sampler2D u_vialB;
uniform float u_t;
uniform vec2 u_resolution;

void main() {
    vec2 uv = gl_FragCoord.xy / u_resolution;
    vec2 state = texture2D(u_state, uv).rg;
    float B = state.g;
    
    // Source image blend
    vec3 srcColor = mix(
        texture2D(u_vialA, uv).rgb,
        texture2D(u_vialB, uv).rgb,
        u_t
    );
    
    // RD organic color (teal/pink vial palette)
    vec3 rdColor = mix(
        vec3(0.04, 0.82, 1.0),   // teal (A-dominant)
        vec3(1.0, 0.38, 0.63),   // pink (B-dominant)
        B
    );
    float brightness = 0.08 + state.r * 0.15 + B * 0.8;
    rdColor *= brightness;
    
    // Composite: 60% source + 40% RD
    vec3 final = srcColor * 0.6 + rdColor * 0.4;
    
    gl_FragColor = vec4(final, 1.0);
}
'''

# Save shaders
shader_dir = os.path.join(output_dir, 'shaders')
os.makedirs(shader_dir, exist_ok=True)
with open(os.path.join(shader_dir, 'rd_sim.glsl'), 'w') as f:
    f.write(shader_code)
with open(os.path.join(shader_dir, 'rd_render.glsl'), 'w') as f:
    f.write(render_shader)

print('Shaders saved:')
print(f'  {shader_dir}/rd_sim.glsl — Gray-Scott simulation')
print(f'  {shader_dir}/rd_render.glsl — Vial palette renderer')
print(f'\nDrop these into the portal as a WebGL post-process on the vial animator.')
print(f'The RD runs per-frame on GPU, parameters driven by the vial sprite sheet.')
print(f'\nOr use the pre-computed RD sprite sheet for CPU-only playback:')
print(f'  {rd_sheet_path}')

In [ ]:
#@title 6. Physarum Agent Overlay (Optional — Mycelium Tendrils)

class PhysarumSim:
    """
    Physarum polycephalum agent simulation.
    Agents deposit trails, follow gradients, form organic networks.
    Used here as a tendril overlay between vial states.
    """
    
    def __init__(self, width, height, n_agents=50000):
        self.w = width
        self.h = height
        self.n = n_agents
        
        # Agent state: x, y, heading
        self.x = np.random.rand(n_agents).astype(np.float32) * width
        self.y = np.random.rand(n_agents).astype(np.float32) * height
        self.heading = np.random.rand(n_agents).astype(np.float32) * 2 * np.pi
        
        # Trail map
        self.trail = np.zeros((height, width), dtype=np.float32)
        
        # Parameters
        self.sensor_dist = 9.0
        self.sensor_angle = np.pi / 4  # 45 degrees
        self.rotation_angle = np.pi / 4
        self.move_speed = 1.0
        self.deposit = 5.0
        self.decay = 0.9
    
    def sense(self, angle_offset):
        """Sample trail at sensor position."""
        angles = self.heading + angle_offset
        sx = (self.x + np.cos(angles) * self.sensor_dist).astype(int) % self.w
        sy = (self.y + np.sin(angles) * self.sensor_dist).astype(int) % self.h
        return self.trail[sy, sx]
    
    def step(self):
        # Sense left, center, right
        sl = self.sense(-self.sensor_angle)
        sc = self.sense(0)
        sr = self.sense(self.sensor_angle)
        
        # Rotate toward strongest signal
        turn_left = (sl > sc) & (sl > sr)
        turn_right = (sr > sc) & (sr > sl)
        random_turn = (sl == sr) & (sl > sc)  # equal: random
        
        self.heading -= turn_left * self.rotation_angle
        self.heading += turn_right * self.rotation_angle
        self.heading += random_turn * (np.random.rand(self.n) - 0.5) * self.rotation_angle * 2
        
        # Move
        self.x = (self.x + np.cos(self.heading) * self.move_speed) % self.w
        self.y = (self.y + np.sin(self.heading) * self.move_speed) % self.h
        
        # Deposit trail
        ix = self.x.astype(int)
        iy = self.y.astype(int)
        np.add.at(self.trail, (iy, ix), self.deposit)
        
        # Diffuse (3x3 mean blur) + decay
        from scipy.ndimage import uniform_filter
        self.trail = uniform_filter(self.trail, size=3) * self.decay
    
    def get_image(self):
        """Return trail map as teal-colored image."""
        t = np.clip(self.trail / self.trail.max(), 0, 1)
        r = t * 0.0
        g = t * 0.82
        b = t * 1.0
        return np.stack([r, g, b], axis=-1)

# Run physarum on first key frame
phy = PhysarumSim(FRAME_W * 2, FRAME_H * 2, n_agents=30000)

# Seed agents on bright regions of the vial
lum = 0.299 * key_frames[0][:,:,0] + 0.587 * key_frames[0][:,:,1] + 0.114 * key_frames[0][:,:,2]
lum_big = np.array(Image.fromarray((lum * 255).astype(np.uint8)).resize((FRAME_W*2, FRAME_H*2))) / 255.0
phy.trail = lum_big * 10  # seed trail from image

print('Running 200 physarum steps...')
for i in range(200):
    phy.step()
    if i % 50 == 0:
        print(f'  Step {i}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(key_frames[0])
ax1.set_title('Source Vial')
ax1.axis('off')
ax2.imshow(phy.get_image())
ax2.set_title('Physarum Network (200 steps)')
ax2.axis('off')
plt.suptitle('Mycelium tendrils grown from vial luminance')
plt.tight_layout()
plt.show()

print('\nThe physarum trails can overlay the RD transitions')
print('as organic tendril connections between vial states.')

In [ ]:
#@title 7. Save Everything
import shutil

# Save summary
summary = {
    'n_captures': n_captures,
    'n_key_frames': len(key_frames),
    'n_rd_frames': len(all_rd_frames),
    'interp_frames_per_pair': INTERP_FRAMES,
    'rd_steps_per_frame': STEPS_PER_FRAME,
    'frame_size': [FRAME_W, FRAME_H],
    'rd_sheet_size': list(rd_sheet.shape[:2]),
    'pairs_processed': N_PAIRS,
}

with open(os.path.join(output_dir, 'manifest.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('=== Output ===')
print(f'RD sprite sheet: {rd_sheet_path}')
print(f'GIF preview: {gif_path}')
print(f'WebGL shaders: {shader_dir}/')
print(f'Individual frames: {output_dir}/')
print(f'Manifest: {output_dir}/manifest.json')
print(f'\n{json.dumps(summary, indent=2)}')
print(f'\n--- Guinea Pig Trench LLC ---')